# Ammonia synthesis equilibrium


**Evidence: numerical verification**, plus source-parameter predictions.
Study N₂ + 3 H₂ ⇌ 2 NH₃ at **573.15 K**, one declared vapor phase, feed
1 mol N₂ + 3 mol H₂, and pressures 100, 1000, 100,000 and 20,000,000 Pa.

Pure parameters come from Gross and Sadowski (2001), Table 2, nitrogen
([paper](https://doi.org/10.1021/ie0003887)); Ghosh, Chapman and French (2003),
Table 1, p. 233, hydrogen (`hda-pc-saft/1`); and Esper et al. (2023), pinned
FeOs `esper2023.json[1714]`, ammonia
([paper](https://doi.org/10.1021/acs.iecr.3c02255)). These are published pure
records assembled for teaching, without a fitted ternary parameter set.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import json
import math
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Mixture,
    GAS_CONSTANT_J_PER_MOL_K,
)
from epcsaft import equilibrium as eq
from epcsaft import ThermochemistryRecord, IdealInterval, IdealCorrelation, IdealShomate

# Start the kernel anywhere within the documented repository checkout.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


## Assemble the three canonical pure records

Read the named components from their packets and preserve the coefficient
locators. All three unlike binary interaction parameters **kᵢⱼ = 0** are an
explicit teaching assumption. Ammonia's source dipole coordinates are zero;
omit that zero contribution exactly and retain its donor/acceptor association
pair (energy/k 200 K, volume 0.0001). The resulting model is nonpolar with
general-site association. No cross-site pair exists on N₂ or H₂.

Nitrogen was fitted at 63–126 K, so **573.15 K is extrapolation**. Hydrogen's
source basis is about 462.5–583.5 K and 1.5–247 bar; the dilute-pressure checks
are outside that pressure fit range. Esper's retained ammonia row asserts no
per-record fit interval. Its SI attribution and CC BY-NC use basis remain
with the packet; this notebook does not relicense those sources.


In [2]:
from copy import deepcopy
from itertools import combinations

sources = [
    ("gross-sadowski-2001/3", "nitrogen"),
    ("hda-pc-saft/1", "hydrogen"),
    ("feos-esper-2023/1", "ammonia"),
]
components = []
for packet, species in sources:
    document = json.loads((PACKETS / packet / "parameters/parameters.json").read_text())
    component = deepcopy(
        next((c for c in document["components"] if c["component_id"] == species))
    )
    if species == "ammonia":
        polar = [
            c
            for c in component["coefficients"]
            if c["family"] in ("dipole_moment", "dipole_count")
        ]
        assert all((c["value"]["magnitude"] == 0 for c in polar))
        component["coefficients"] = [
            c for c in component["coefficients"] if c not in polar
        ]
        sites = [
            s for s in document["topology"]["sites"] if s["component_id"] == species
        ]
        edges = [
            e
            for e in document["topology"]["edges"]
            if e["endpoint_a"]["component_id"] == species
            and e["endpoint_b"]["component_id"] == species
        ]
    components.append(component)
pairs = [
    {
        "component_id_a": a,
        "component_id_b": b,
        "coefficients": [
            {
                "identity": f"pair/{a}/{b}/k_ij",
                "family": "k_ij",
                "value": {"magnitude": 0.0, "unit": "dimensionless"},
            }
        ],
    }
    for a, b in combinations([c["component_id"] for c in components], 2)
]
mapping = dict(
    schema="epcsaft.parameters",
    schema_version=1,
    components=components,
    correlations=[],
    pairs=pairs,
    model_coefficients=[],
    topology=dict(sites=sites, edges=edges, no_cross_association=[]),
    model_families=[f for f in document["model_families"] if f["kind"] != "polar"],
)
parameters = Parameters.from_mapping(mapping)
model = Mixture(parameters)
display(
    pd.DataFrame(
        [
            {
                "Species": c["component_id"],
                **{v["family"]: v["value"]["magnitude"] for v in c["coefficients"]},
            }
            for c in components
        ]
    )
)

charges = [c["fixed"]["charge_number"]["value"]["magnitude"] for c in components]
assert charges == [0, 0, 0]


,Species,dispersion_energy_over_k,segment_count,segment_diameter
0,nitrogen,90.96000,1.20530,3.3130
1,hydrogen,19.27750,1.00000,2.9860
2,ammonia,213.57063,2.44608,2.3709


## Check the thermochemical scalar and solve at fixed temperature

The gas Shomate rows are from the NIST Chemistry WebBook:
[N₂](https://webbook.nist.gov/cgi/cbook.cgi?ID=C7727379&Type=JANAFG),
[H₂](https://webbook.nist.gov/cgi/cbook.cgi?ID=C1333740&Type=JANAFG),
[NH₃](https://webbook.nist.gov/cgi/cbook.cgi?ID=C7664417&Mask=1), citing Chase
(1998). The intervals are 500–2000 K, 298–1000 K and 298–1400 K, respectively.
The coefficient rows below retain A–H and set formation enthalpy to 1000 H
J/mol. The standard pressure is P° = 100,000 Pa. Chase's Zotero record is
`CWCVM9IE`, snapshot `VXZKCLGY`; no full Chase PDF is claimed.

The native thermal model evaluates ideal h − Ts at density P°/(RT).
Verify the declared ln Kp = −5.4047677937089755 within 10⁻⁸ J/mol on the
reaction Gibbs energy. For Δν = −2 and ρ° = 1 mol/m³, check

$$\ln K_\rho = \ln K_p - \Delta\nu\ln(RT\rho^\circ/P^\circ)$$

within 10⁻¹² in ln K. Supply only the fixed scalar Kρ to the native reaction
solve. A constant Kρ and the Shomate temperature derivatives would be
inconsistent if both were supplied as reaction standards. Keep T fixed;
do not request reacting caloric derivatives from this declaration.

Python only checks balances and the independent mass-action relation
Σνᵢ[ln(xᵢP/P°) + ln φᵢ] − ln Kp. Require atom balances within 10⁻¹⁰ mol,
positive amounts, normalized fractions within 10⁻¹², positive pressure slope,
pressure closure within 10⁻¹⁰ relative, classified residuals at most 10⁻¹⁰
and independent mass action within 10⁻⁹. All species are uncharged.

For the independent analytic ideal comparison, use
$a=\sqrt{27K_p}(P/P^\circ)/16$ and
$y=2a/(1+2a+\sqrt{1+4a})$.
Require relative agreement within 10⁻⁴ at 100/1000 Pa and a visible difference
above 2% at 200 bar. This algebra is a verification, not a Python root find.


In [3]:
T = 573.15
p0 = 100000.0
R = GAS_CONSTANT_J_PER_MOL_K
ln_Kp = -5.4047677937089755
ln_Krho = -11.492330833421008
coefs = [
    [19.50583, 19.88705, -8.598535, 1.369784, 0.527601, -4.935202, 212.39, 0],
    [33.066178, -11.363417, 11.432816, -2.772874, -0.158558, -9.980797, 172.707974, 0],
    [19.99563, 49.77119, -15.37599, 1.921168, 0.189174, -53.30667, 203.8591, -45.89806],
]
thermal = ThermochemistryRecord(
    p0,
    [
        [
            IdealInterval(
                lo, hi, True, True, IdealCorrelation(IdealShomate(c, c[-1] * 1000), p0)
            )
        ]
        for c, (lo, hi) in zip(coefs, [(500, 2000), (298, 1000), (298, 1400)])
    ],
)
reference = Mixture(parameters, thermochemistry=thermal)
gibbs = []
for i in range(3):
    x = [0.0, 0.0, 0.0]
    x[i] = 1.0
    state = reference.state(T, rho=p0 / (R * T), x=x)
    gibbs.append(state.ideal_enthalpy - T * state.ideal_entropy)
thermal_error = sum((nu * g for nu, g in zip((-1.0, -3.0, 2.0), gibbs))) + R * T * ln_Kp
assert abs(thermal_error) < 1e-08
assert abs(ln_Krho - ln_Kp - 2 * math.log(R * T / p0)) < 1e-12
rows = []
for P in (100.0, 1000.0, 100000.0, 20000000.0):
    problem = eq.Problem(
        T=T,
        P=P,
        phases=[eq.Phase("gas", kind="vapor", composition_guess=(0.2, 0.6, 0.2))],
        feed=eq.Amounts({"nitrogen": 1.0, "hydrogen": 3.0}),
        reactions=[
            eq.Reaction(
                {"nitrogen": -1.0, "hydrogen": -3.0, "ammonia": 2.0},
                equilibrium_constant=1.0208081643306505e-05,
            )
        ],
    )
    assert abs(math.log(problem.reactions[0].equilibrium_constant) - ln_Krho) < 1e-12
    result = eq.solve_equilibrium(model, problem)
    assert result.success, result.message
    x = result.mole_fractions
    n = [result.amounts[0] * v for v in x]
    element_errors = [2 * n[0] + n[2] - 2.0, 2 * n[1] + 3 * n[2] - 6.0]
    state = model.state(T, rho=result.molar_densities[0], x=x)
    lnphi = state.log_fugacity_coefficient
    law = (
        sum(
            (
                nu * (math.log(v * P / p0) + f)
                for nu, v, f in zip((-1.0, -3.0, 2.0), x, lnphi)
            )
        )
        - ln_Kp
    )
    a = math.sqrt(math.exp(ln_Kp)) * P / p0 * math.sqrt(27.0) / 16.0
    ideal = 2 * a / (1 + 2 * a + math.sqrt(1 + 4 * a))
    classified = max((abs(v) for v in result.classified_residuals))
    assert max((abs(v) for v in element_errors)) < 1e-10
    assert (
        abs(sum(x) - 1) < 1e-12
        and min(n) > 0
        and (state.pressure_density_derivative > 0)
    )
    assert abs(law) < 1e-09 and classified < 1e-10
    assert abs(state.pressure / P - 1) < 1e-10
    if P <= 1000:
        assert abs(x[2] / ideal - 1) < 0.0001
    rows.append(
        dict(
            P_Pa=P,
            amounts_mol=n,
            mole_fractions=x,
            ideal_ammonia_fraction=ideal,
            relative_ideal_difference=x[2] / ideal - 1,
            ln_phi=lnphi,
            element_balance_error_mol=element_errors,
            charge_balance_mol=sum(z * amount for z, amount in zip(charges, n)),
            ln_mass_action_residual=law,
            classified_residual=classified,
            pressure_closure_relative=state.pressure / P - 1,
        )
    )
assert abs(rows[-1]["relative_ideal_difference"]) > 0.02
values = dict(
    T_K=T,
    ln_Kp=ln_Kp,
    ln_Krho=ln_Krho,
    thermal_gibbs_J_mol=gibbs,
    thermal_error_J_mol=thermal_error,
    rows=rows,
)
values["standard_conversion_error"] = ln_Krho - ln_Kp - 2 * math.log(R * T / p0)
display(
    pd.DataFrame(rows)[
        [
            "P_Pa",
            "ideal_ammonia_fraction",
            "relative_ideal_difference",
            "classified_residual",
            "ln_mass_action_residual",
            "pressure_closure_relative",
        ]
    ].assign(ammonia_mole_fraction=[row["mole_fractions"][2] for row in rows])
)


,P_Pa,ideal_ammonia_fraction,relative_ideal_difference,classified_residual,ln_mass_action_residual,pressure_closure_relative,ammonia_mole_fraction
0,100.0,0.000022,8.692937e-07,6.661048e-16,-1.243450e-14,4.440892e-16,0.000022
1,1000.0,0.000218,8.693484e-06,6.658439e-16,-7.993606e-15,6.661338e-16,0.000218
2,100000.0,0.020874,8.743861e-04,2.795053e-16,0.000000e+00,6.661338e-16,0.020892
3,20000000.0,0.622052,1.183401e-01,2.459158e-16,3.552714e-15,-3.330669e-16,0.695665


In [4]:
display(pd.DataFrame(rows)[[
    "P_Pa", "amounts_mol", "mole_fractions", "ln_phi",
    "element_balance_error_mol", "charge_balance_mol",
]].rename(columns={
    "P_Pa": "Pressure / Pa",
    "amounts_mol": "N2, H2, NH3 amounts / mol",
    "mole_fractions": "N2, H2, NH3 fractions / 1",
    "ln_phi": "N2, H2, NH3 ln fugacity coefficients / 1",
    "element_balance_error_mol": "N, H atom balance errors / mol",
    "charge_balance_mol": "Charge balance / mol",
}))
display(pd.DataFrame([{
    "Temperature / K": values["T_K"],
    "ln Kp / 1": values["ln_Kp"],
    "ln Krho / 1": values["ln_Krho"],
    "Thermal reaction error / J mol^-1": values["thermal_error_J_mol"],
    "Standard conversion error / 1": values["standard_conversion_error"],
}]))
display(pd.DataFrame({
    "Species": ["nitrogen", "hydrogen", "ammonia"],
    "Ideal standard Gibbs energy / J mol^-1": values["thermal_gibbs_J_mol"],
}))


,Pressure / Pa,"N2, H2, NH3 amounts / mol","N2, H2, NH3 fractions / 1","N2, H2, NH3 ln fugacity coefficients / 1","N, H atom balance errors / mol",Charge balance / mol
0,100.0,"[0.9999564554816837, 2.99986936644505, 8.708903663315427e-05]","[0.24999455681670008, 0.7499836704501001, 2.177273319987838e-05]","[4.915850051563298e-07, 4.475280941702228e-07, 4.775341711162814e-08]","[4.440892098500626e-16, 0.0]",0.0
1,1000.0,"[0.9995648072185193, 2.998694421655558, 0.0008703855629615834]","[0.2499455890626913, 0.749836767188074, 0.0002176437492346965]","[4.914312311222707e-06, 4.475794198607798e-06, 4.736158619205332e-07]","[0.0, 8.881784197001252e-16]",0.0
2,100000.0,"[0.9590703402425576, 2.8772110207276733, 0.08185931951488447]","[0.244776902733317, 0.7343307081999512, 0.02089238906673184]","[0.0004757427142311511, 0.00045339169951193317, 6.672124227196719e-06]","[-4.440892098500626e-16, 0.0]",0.0
3,20000000.0,"[0.17947806850656361, 0.5384342055196899, 1.641043862986873]","[0.07608368196867617, 0.2282510459060281, 0.6956652721252957]","[0.1076214620231393, 0.22346882100238727, -0.15609040140853978]","[4.440892098500626e-16, -8.881784197001252e-16]",0.0


,Temperature / K,ln Kp / 1,ln Krho / 1,Thermal reaction error / J mol^-1,Standard conversion error / 1
0,573.15,-5.404768,-11.492331,0.0,8.881784e-16


,Species,Ideal standard Gibbs energy / J mol^-1
0,nitrogen,-112733.235745
1,hydrogen,-77796.786453
2,ammonia,-160183.764774


## Meaning and limits


The dilute limit verifies the mathematical reduction to ideal gas behavior.
The 200 bar result is a prediction from the declared source parameters and
zero unlike interactions, with unquantified extrapolation and parameter/model
uncertainty. It does not validate high-pressure ammonia accuracy. One declared
vapor phase does not establish global stability. Kinetics, catalyst effects,
condensation, reactor energy balance and industrial performance are outside
this isothermal example.
